In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display

from analysis.src.data import complete_case_mask
from analysis.src.io import experiment_paths, load_data
from analysis.src.plotting import configure_plotting
from analysis.src.constants import COVARIATES, OUTCOMES
from paths import paths

In [ ]:
configure_plotting()

In [ ]:
OUT = experiment_paths('robustness/walkshed_duration')
DISEASES = OUTCOMES

data = load_data()

In [ ]:
# Compare walkability across walkshed durations
datasets = {15: data.copy()}

for minutes in [10, 20]:
    alternative = gpd.read_parquet(paths.results.by_duration[minutes])
    datasets[minutes] = data.assign(walkability_score = alternative.walkability_score.reindex(data.index))

data_10, data_15, data_20 = (datasets[minutes] for minutes in [10, 15, 20])

walk_scores = pd.DataFrame(
    {'10 min': data_10.walkability_score, '15 min': data_15.walkability_score, '20 min': data_20.walkability_score}
)
walk_scores.corr(method = 'spearman').rename_axis('walkshed').reset_index().to_csv(
    OUT.walkshed_rank_correlations_csv, index = False
)

In [ ]:
# Fit association models for each walkshed duration
results = []
for outcome in DISEASES:
    for mins, dataset in zip([10, 15, 20], [data_10, data_15, data_20]):
        data = dataset.loc[complete_case_mask(dataset, outcome, COVARIATES)]

        formula = f'{outcome} ~ walkability_score + ' + ' + '.join(COVARIATES)
        model = smf.ols(formula, data = data).fit()

        results.append(
            {
                'outcome': outcome,
                'mins': mins,
                'coef': model.params['walkability_score'],
                'ci_lower': model.conf_int().loc['walkability_score', 0],
                'ci_upper': model.conf_int().loc['walkability_score', 1],
                'r2': model.rsquared,
                'aic': model.aic,
                'bic': model.bic
            }
        )

results_df = pd.DataFrame(results)

display(results_df.head())

In [ ]:
_, axes = plt.subplots(2, 4, figsize = (18, 10))
axes = axes.flatten()

for i, outcome in enumerate(DISEASES):
    if i >= len(axes):
        break

    ax = axes[i]
    subset = results_df[results_df['outcome'] == outcome]

    ax.errorbar(
        subset['mins'],
        subset['coef'],
        yerr = [subset['coef'] - subset['ci_lower'], subset['ci_upper'] - subset['coef']],
        fmt = '-o',
        capsize = 5,
        markersize = 8,
    )
    ax.axhline(0, color = 'red', linestyle = '--')
    ax.set_title(f'Coefficient for {outcome}')
    ax.set_xlabel('Walkshed Duration (minutes)')
    ax.set_ylabel('Coefficient')
    ax.set_xticks([10, 15, 20])

plt.tight_layout()
plt.savefig(OUT.walkshed_sensitivity_png, dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
# Summarise associations per standard deviation of walkability
duration_sd = {
    minutes: frame.walkability_score.std(ddof = 1) for minutes, frame in [(10, data_10), (15, data_15), (20, data_20)]
}
results_df['walkability_sd'] = results_df['mins'].map(duration_sd)
results_df.to_csv(OUT.walkshed_models_csv, index = False)